# 03 — Token Efficiency: Monolithic vs ICM-Staged Context



## Objective

Compare monolithic vs. ICM-staged context loading. Report token throughput,
VRAM utilization, and attention degradation metrics.

The ICM methodology (see `docs/research.md`) divides context into 5 layers:
* L0 system (256 tok, pinned)
* L1 persona (512 tok, pinned)
* L2 instructions (1024 tok, HBM)
* L3 reference (8192 tok, HBM-paged)
* L4 working (4096 tok, HBM-working)


In [ ]:
# --- Kaggle / fresh-runtime setup ---
import os, shutil, sys, subprocess
from pathlib import Path

SETUP_LOG = Path('/kaggle/working/nexusrt_setup.log')


def _is_repo_root(root: Path) -> bool:
    return (root / 'packaging' / 'pyproject.toml').exists() and (root / 'python' / 'nexusrt').exists()


def _find_repo_root() -> Path:
    input_candidates = [Path('/kaggle/input/nexusrt'), Path('/kaggle/input/nexusrt/nexusrt')]
    if Path('/kaggle/input').exists():
        input_candidates.extend(p.parent.parent for p in Path('/kaggle/input').glob('**/packaging/pyproject.toml'))
    for root in input_candidates:
        if _is_repo_root(root):
            return root

    working_candidates = [Path.cwd(), Path('/kaggle/working/nexusrt')]
    if Path('/kaggle/working').exists():
        working_candidates.extend(p.parent.parent for p in Path('/kaggle/working').glob('**/packaging/pyproject.toml'))
    for root in working_candidates:
        if _is_repo_root(root):
            return root
    raise RuntimeError('NexusRT repo not found. Attach the private Kaggle dataset rbrtsl/nexusrt first.')


def _tail(text: str, n: int = 120) -> str:
    lines = text.splitlines()
    return '\n'.join(lines[-n:])


def _run(cmd, *, check=True, env=None):
    cmd = [str(x) for x in cmd]
    header = '+ ' + ' '.join(cmd)
    print(header)
    with SETUP_LOG.open('a') as f:
        f.write('\n' + header + '\n')
    proc = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env)
    if proc.stdout:
        with SETUP_LOG.open('a') as f:
            f.write(proc.stdout)
        print(_tail(proc.stdout, 80))
    if check and proc.returncode:
        print(f'Command failed with exit code {proc.returncode}. Full log: {SETUP_LOG}')
        if proc.stdout:
            print('--- last setup log lines ---')
            print(_tail(proc.stdout, 160))
        raise subprocess.CalledProcessError(proc.returncode, cmd, output=proc.stdout)
    return proc


SETUP_LOG.write_text('NexusRT Kaggle setup log\n')
print('NexusRT Kaggle setup: direct CUDA runtime refresh v4')
src_repo = _find_repo_root().resolve()
work_repo = Path('/kaggle/working/nexusrt')
if str(src_repo).startswith('/kaggle/input/'):
    if work_repo.exists():
        shutil.rmtree(work_repo)
    shutil.copytree(src_repo, work_repo, ignore=shutil.ignore_patterns('.venv', 'packaging/build', '__pycache__', '.pytest_cache'))
    repo = work_repo
else:
    repo = src_repo

os.chdir(repo)
print(f'NexusRT repo: {repo}')
_run(['nvidia-smi'], check=False)
_run(['nvcc', '--version'], check=False)

# Keep source imports available even if editable install fails.
python_dir = str(repo / 'python')
if python_dir not in sys.path:
    sys.path.insert(0, python_dir)
os.environ['PYTHONPATH'] = python_dir + os.pathsep + os.environ.get('PYTHONPATH', '')
os.environ.setdefault('NEXUSRT_BOOT_TRACE', '1')

# Kaggle validation uses direct source imports plus the locally built C++ core.
# This avoids editable-install/build-isolation failures in private dataset runs.
print('Using source import + direct CUDA CMake build; skipping editable pip install.')

# Build the CUDA shared library directly so nexusrt._abi can load it.
build_dir = repo / 'packaging' / 'build'
configure_cmd = [
    'cmake', '-S', 'packaging', '-B', str(build_dir), '-GNinja',
    '-DCMAKE_BUILD_TYPE=Release',
    '-DNEXUSRT_ENABLE_CUDA=ON',
    '-DNEXUSRT_ENABLE_METAL=OFF',
    '-DNEXUSRT_BUILD_TESTS=OFF',
    '-DCMAKE_CUDA_ARCHITECTURES=60;75;80;90',
]
configure_ok = _run(configure_cmd, check=False).returncode == 0
if not configure_ok:
    raise RuntimeError(f'CUDA CMake configure failed. These notebooks require CUDA; see {SETUP_LOG}')

build_ok = _run(['cmake', '--build', str(build_dir), '--parallel', '2'], check=False).returncode == 0
if not build_ok:
    raise RuntimeError(f'CUDA CMake build failed. These notebooks require CUDA; see {SETUP_LOG}')

lib_candidates = [
    build_dir / 'outputs' / 'lib' / 'libnexusrt.so',
    build_dir / 'outputs' / 'lib' / 'libnexusrt.dylib',
]
for lib in lib_candidates:
    if lib.exists():
        os.environ['NEXUSRT_LIB'] = str(lib)
        print(f'NEXUSRT_LIB={lib}')
        break
else:
    raise RuntimeError(f'Built NexusRT, but libnexusrt was not found under {build_dir}/outputs/lib. See {SETUP_LOG}')
import nexusrt
info = nexusrt.firmware.init('auto')
print(f'Vendor      : {info.vendor}')
print(f'Arch        : {info.arch}')
print(f'Name        : {info.name}')
print(f'HBM (GB)    : {info.hbm_bytes >> 30}')
print(f'SMs         : {info.sm_count}')
print(f'Features    : {info.features}')



In [ ]:

# Baseline: monolithic context — load everything as one big allocation.
import time
import nexusrt.memory as mem
TOTAL_TOKENS = 32 * 1024  # 32k tokens
TOKEN_BYTES = 4  # int32

mono = mem.alloc(TOTAL_TOKENS * TOKEN_BYTES, read_mostly=True)
print(f"Monolithic allocation: {TOTAL_TOKENS * TOKEN_BYTES >> 20} MB")

# Simulate decoding with monolithic context: every step touches all tokens.
t0 = time.perf_counter()
for step in range(64):
    mem.prefetch(mono.ptr, mono.bytes)
t1 = time.perf_counter()
mono_per_step_us = (t1 - t0) * 1e6 / 64
print(f"Monolithic: {mono_per_step_us:.1f} us/step")
mem.free(mono)


In [ ]:

# ICM-staged: scope to L4_working only for the decode loop, prefetch L3
# slots on demand via attention hints.
import nexusrt.token_opt as tk

budget = tk.context_scope("infer",
    [tk.IcmLayer.L0_SYSTEM, tk.IcmLayer.L1_PERSONA,
     tk.IcmLayer.L2_INSTRUCTIONS, tk.IcmLayer.L3_REFERENCE,
     tk.IcmLayer.L4_WORKING])
print(f"ICM total budget: {budget} tokens (vs monolithic {TOTAL_TOKENS} tokens)")

# Allocate a small L4 KV cache.
L4_TOKENS = 4096
l4 = mem.alloc(L4_TOKENS * 4096, read_mostly=False)  # 4KB/slot
print(f"L4 KV cache: {L4_TOKENS * 4096 >> 20} MB")

# Simulate attention-weighted prefetch — only top-32 slots per step.
import random
t0 = time.perf_counter()
for step in range(64):
    topk = random.sample(range(L4_TOKENS), 32)
    tk.prefetch_attention(l4.ptr, L4_TOKENS, topk)
t1 = time.perf_counter()
icm_per_step_us = (t1 - t0) * 1e6 / 64
print(f"ICM-staged: {icm_per_step_us:.1f} us/step")
print(f"Speedup: {mono_per_step_us / icm_per_step_us:.2f}x")
print(f"Token reduction: {TOTAL_TOKENS / L4_TOKENS:.1f}x")

mem.free(l4)


In [ ]:

nexusrt.firmware.shutdown()
print("✓ Token efficiency test complete.")
